# Volume 13, Chapter 4: The large sieve and Bombieri–Vinogradov (Python companion)

Exercises first, solutions at the end. We check Vaughan's identity and watch its four pieces cancel, test the large sieve for characters with fast Fourier transforms, measure the Bombieri–Vinogradov sum and Hooley's variance formula, and count least quadratic non-residues. The computer science application: **computing square roots modulo a prime** with the Tonelli–Shanks algorithm, which needs one quadratic non-residue, and Linnik's theorem explains why the naive search for it is fast. Requires `numpy` and `sympy`.

In [ ]:
import math, random, time
import numpy as np
from sympy import primerange, primitive_root, randprime, isprime
random.seed(4)

def spf_sieve(N):
    spf = np.zeros(N + 1, dtype=np.int32)
    for i in range(2, int(N**0.5) + 1):
        if spf[i] == 0:
            block = spf[i*i::i]; block[block == 0] = i; spf[i*i::i] = block
    return spf

def mangoldt(N):
    # Lambda(n) for n <= N
    spf = spf_sieve(N)
    lam = np.zeros(N + 1)
    for n in range(2, N + 1):
        p = spf[n] if spf[n] else n
        m = n
        while m % p == 0:
            m //= p
        if m == 1:
            lam[n] = math.log(p)
    return lam

def mobius(N):
    mu = np.ones(N + 1, dtype=np.int64); mu[0] = 0
    is_comp = np.zeros(N + 1, bool)
    for p in range(2, N + 1):
        if not is_comp[p]:
            is_comp[2*p::p] = True
            mu[p::p] *= -1
            mu[p*p::p*p] = 0
    return mu

## Exercises

**P4.1 (★) Vaughan's identity.** With $U=V=10$, compute $a_1,a_2,a_3,a_4$ of Lemma 4.3 for $n\le 3000$ directly from their definitions, and check that they add up to $\Lambda(n)$. Then, for $x=3000$, print $\sum_{n\le x}a_i(n)$ for each $i$ and compare with $\psi(x)$. Which pieces are large, and how much do they cancel?

**P4.2 (★★) The large sieve for characters.** Let $N=10^4$ and $Q=100$. For prime moduli $p\le Q$ every non-principal character is primitive, and the characters are $\chi_j(n)=e\bigl(j\operatorname{ind}(n)/(p-1)\bigr)$, where $\operatorname{ind}$ is the discrete logarithm to a primitive root. Compute $\sum_{p\le Q}\frac p{p-1}\sum_{j=1}^{p-2}|\sum_{n\le N}a_n\chi_j(n)|^2$ (plus the term $q=1$) with one FFT per prime, for (a) random signs $a_n=\pm1$, (b) $a_n=\Lambda(n)$, (c) $a_n=1$ for $n\equiv1\pmod{30}$. Compare with $(\pi N+Q^2)\sum|a_n|^2$ from Theorem 4.1, which is a bound for the sum over all moduli $q\le Q$, not only the primes.

**P4.3 (★★) Bombieri–Vinogradov and Hooley, in numbers.** For $x=10^6$, compute $E(x;q)=\max_{(a,q)=1}|\theta(x;q,a)-\frac x{\varphi(q)}|$ and $V(q)=\sum_{(a,q)=1}(\theta(x;q,a)-\frac x{\varphi(q)})^2$ for $q\le 4000$. Print $\frac1x\sum_{q\le Q}E(x;q)$ and the trivial bound $\sum_{q\le Q}\frac1{\varphi(q)}$ at $Q=100,1000,4000$, and compare $\frac1{Qx}\sum_{q\le Q}V(q)$ with Hooley's $\log Q-c$, $c=\gamma+\log2\pi+1+\sum_p\frac{\log p}{p(p-1)}$ (Remark 4.5).

**P4.4 (★★) Square roots modulo $p$ (computer science).** (a) Compute $n_2(p)$ for all odd primes $p<10^6$; compare the proportion with $n_2(p)=p_k$ with $2^{-k}$, the mean with Erdős's constant $\sum_kp_k2^{-k}$ (Exercise 4.5), and count the $p$ with $n_2(p)>10^{6\varepsilon}$ for $\varepsilon=0.2,0.25$. (b) Implement Tonelli–Shanks, finding the non-residue $z$ by trying $2,3,5,7,\dots$, and test it on $1000$ random squares modulo random $256$-bit primes $p\equiv1\pmod{2^{20}}$, the hard case where $p-1$ has a large power of $2$. How many candidates $z$ were needed, at most?

## Solutions

### P4.1
Each piece is computed straight from its definition as a divisor sum; $c(m)=\sum_{b\mid m,\,b>V}\mu(b)$.

In [ ]:
X = 3000; U = V = 10
lam = mangoldt(X); mu = mobius(X)
a = np.zeros((5, X + 1))
for n in range(1, X + 1):
    if n <= U:
        a[1][n] = lam[n]
for b in range(1, V + 1):
    for d in range(1, X // b + 1):
        a[2][b*d] += mu[b] * math.log(d)
for b in range(1, V + 1):
    for c in range(1, U + 1):
        if lam[c] == 0: continue
        for d in range(1, X // (b*c) + 1):
            a[3][b*c*d] -= mu[b] * lam[c]
cm = np.zeros(X + 1)
for b in range(V + 1, X + 1):
    cm[b::b] += mu[b]
for k in range(U + 1, X + 1):
    if lam[k] == 0: continue
    for m in range(V + 1, X // k + 1):
        a[4][k*m] += lam[k] * cm[m]
err = np.abs(a[1] + a[2] + a[3] + a[4] - lam)[1:].max()
print(f"max |a1+a2+a3+a4 - Lambda| for n <= {X}: {err:.1e}")
sums = [a[i][1:].sum() for i in range(1, 5)]
print("sums of the pieces:", [round(float(s), 1) for s in sums], " psi(x) =", round(float(lam.sum()), 1))
print("max |c(m)| / tau(m):", max(abs(cm[m]) / sum(1 for d in range(1, m + 1) if m % d == 0) for m in range(1, 400)))

**What it shows.** The identity holds to rounding error, $10^{-12}$ or so. The pieces are of very different sizes: $\sum a_1=\psi(10)\approx7.8$, $\sum a_2\approx4252$ overshoots $\psi(3000)\approx3001$ by forty percent, and the bilinear pieces bring it back, $\sum a_3\approx-463$ and $\sum a_4\approx-796$. Also $|c(m)|\le\frac12\tau(m)$ for $m<400$, comfortably within Lemma 4.3. Vaughan's identity does not make anything small; it rearranges $\Lambda$ into pieces whose \emph{twists by characters} can each be bounded, the type I pieces by Pólya–Vinogradov and the type II pieces by the large sieve.

### P4.2
For a prime $p$, group the $a_n$ with $p\nmid n$ by $r=\operatorname{ind}(n)$; then $\sum_na_n\chi_j(n)=\sum_rA_re(jr/(p-1))$ is a discrete Fourier transform of length $p-1$.

In [ ]:
N, Q = 10**4, 100
lamN = mangoldt(N)

def ls_lhs(a):
    tot = q1 = abs(a[1:].sum())**2                 # q = 1
    for p in primerange(2, Q + 1):
        if p == 2: continue                         # no primitive character mod 2
        g = primitive_root(p)
        ind = np.zeros(p, dtype=np.int64); x = 1
        for r in range(p - 1):
            ind[x] = r; x = x * g % p
        n = np.arange(1, N + 1); keep = n % p != 0
        A = np.bincount(ind[n[keep] % p], weights=a[1:][keep], minlength=p - 1)
        F = np.fft.fft(A)                          # F[j] = sum_r A_r e(-jr/(p-1)): chi_j-bar sums, same moduli
        tot += p / (p - 1) * (np.abs(F[1:])**2).sum()
    return tot, q1

tests = {"random signs": np.r_[0, np.random.default_rng(4).choice([-1.0, 1.0], N)],
         "Lambda(n)": lamN,
         "n = 1 mod 30": np.r_[0, (np.arange(1, N + 1) % 30 == 1).astype(float)]}
for name, a in tests.items():
    lhs, q1 = ls_lhs(a); rhs = (math.pi * N + Q**2) * (a**2).sum()
    print(f"{name:14s} LHS/RHS = {lhs / rhs:.4f}   of which q = 1: {q1 / rhs:.4f}")

**What it shows.** For random signs the ratio is $0.027$: each of the roughly $1200$ characters sees a sum of size about $\sqrt N$, so the left side is about $1200N$, against $(\pi N+Q^2)N\approx4.1N^2$. For $\Lambda(n)$ the ratio is $0.035$, but most of it, $0.030$, is the term $q=1$, that is $\psi(N)^2$: the twisted sums $\psi(N,\chi)$ themselves are tiny, because the primes are well distributed among the classes modulo each small $p$, which is what Bombieri–Vinogradov is about. For the progression $n\equiv1\pmod{30}$ the ratio is the largest, $0.051$: modulo $p=3$ and $p=5$ the sequence sits in a single class, so the characters modulo $3$ and $5$ have sums as large as the trivial character's. Prime moduli are only part of the left side of Theorem 4.1, but they show the mechanism: a sequence can correlate with a few characters, never with most of them.

### P4.3
One `bincount` per modulus gives all the $\theta(x;q,a)$ at once.

In [ ]:
x = 10**6
P = np.array(list(primerange(2, x + 1))); w = np.log(P)
Qmax = 4000
E = np.zeros(Qmax + 1); Vq = np.zeros(Qmax + 1); invphi = np.zeros(Qmax + 1)
for q in range(1, Qmax + 1):
    s = np.bincount(P % q, weights=w, minlength=q)
    red = np.gcd(np.arange(q), q) == 1 if q > 1 else np.array([True])
    phi = red.sum(); dev = s[red] - x / phi
    E[q] = np.abs(dev).max(); Vq[q] = (dev**2).sum(); invphi[q] = 1 / phi
c = 0.5772156649 + math.log(2 * math.pi) + 1 + sum(math.log(p) / (p * (p - 1)) for p in primerange(2, 10**6))
print(f"Hooley's constant c = {c:.4f}")
for Q in (100, 1000, 4000):
    print(f"Q={Q:5d}  sum E/x = {E[1:Q+1].sum()/x:.4f}   trivial sum 1/phi = {invphi[1:Q+1].sum():.2f}"
          f"   V/(Qx) = {Vq[1:Q+1].sum()/(Q*x):.3f}   log Q - c = {math.log(Q)-c:.3f}")

**What it shows.** $c=4.1705$. The Bombieri–Vinogradov sum is $0.057x$, $0.352x$ and $0.980x$ at $Q=100$, $1000=\sqrt x$ and $4000$, against trivial bounds of $8.9x$, $13.4x$ and $16.2x$ or so: up to $\sqrt x$, the errors are a few percent of the main terms, on average. The variance per modulus, $\frac1{Qx}\sum V$, is $1.34$, $2.93$, $4.19$, against Hooley's $0.44$, $2.74$, $4.12$: the formula is useless at $Q=100$ (it is meant for $Q$ close to $x$) and already within $0.07$ at $Q=4000=\frac x{250}$. Figures 4.1 and 4.2 extend these numbers to $x=10^7$.

### P4.4
Euler's criterion decides residuosity in one exponentiation. Tonelli–Shanks writes $p-1=2^sq$ with $q$ odd and corrects a first guess $a^{(q+1)/2}$ by powers of $z^q$, which has order exactly $2^s$ because $z$ is a non-residue.

In [ ]:
small = list(primerange(2, 200))
def n2(p):
    for l in small:
        if pow(l, (p - 1) // 2, p) == p - 1:
            return l

ps = list(primerange(3, 10**6))
v = np.array([n2(p) for p in ps])
print(f"{len(ps)} odd primes, mean n2 = {v.mean():.4f}, Erdos constant = {sum(p / 2**k for k, p in enumerate(primerange(2, 400), 1)):.4f}")
for k, l in enumerate(small[:8], 1):
    print(f"n2 = {l:2d}: proportion {np.mean(v == l):.5f}   2^-{k} = {2.0**-k:.5f}")
for eps in (0.2, 0.25):
    print(f"eps = {eps}: {np.sum(v > 10**(6*eps))} primes p < 10^6 with n2(p) > 10^(6 eps) = {10**(6*eps):.1f}")
i = v.argmax(); print("largest:", v[i], "at p =", ps[i])

def tonelli_shanks(a, p):
    # a square root of a modulo the odd prime p, assuming a is a non-zero square
    q, s = p - 1, 0
    while q % 2 == 0:
        q //= 2; s += 1
    z, tries = 2, 1
    while pow(z, (p - 1) // 2, p) != p - 1:
        z += 1; tries += 1
    m, c, t, r = s, pow(z, q, p), pow(a, q, p), pow(a, (q + 1) // 2, p)
    while t != 1:
        i, t2 = 0, t
        while t2 != 1:
            t2 = t2 * t2 % p; i += 1
        b = pow(c, 1 << (m - i - 1), p)
        m, c, t, r = i, b * b % p, t * b * b % p, r * b % p
    return r, tries

rng = random.Random(4); worst = 0; t0 = time.time(); count = 0
while count < 1000:
    p = (rng.getrandbits(236) << 20) + 1
    if not isprime(p): continue
    a = pow(rng.randrange(1, p), 2, p)
    r, tries = tonelli_shanks(a, p)
    assert r * r % p == a
    worst = max(worst, tries); count += 1
print(f"1000 square roots modulo random 256-bit p = 1 mod 2^20: all correct, at most {worst} candidates z, {time.time()-t0:.1f}s")

**What it shows.** (a) The $78497$ odd primes below $10^6$ have mean $n_2(p)=3.646$, approaching Erdős's $3.6746$ from below. The proportions with $n_2(p)=2,3,5,7,\dots$ are $0.5004$, $0.2503$, $0.1252$, $0.0627$, and so on, within $2\%$ of $2^{-k}$ up to $k=8$: residuosity of $2,3,5,\dots$ behaves like independent fair coins (Exercise 4.5). There are $1145$ primes with $n_2(p)>10^{1.2}\approx15.8$ and $15$ with $n_2(p)>10^{1.5}\approx31.6$, all much smaller than Burgess's $p^{0.152}$; the record is $n_2(366791)=43$. Linnik's theorem says the second count stays bounded as $x$ grows with $\varepsilon$ fixed; at this height the bound $C(\varepsilon)$ of the proof, about $9/\rho(8)$, is astronomically larger. (b) All $1000$ roots are correct, and the search for $z$ never needed more than $30$ candidates (that is, $n_2(p)\le31$ for all $1000$ primes), and the whole run takes a few seconds. A randomised search would succeed with probability $\frac12$ per try; the deterministic search is not proved fast, but GRH (Bach) bounds it by $2\log^2p$ steps, and Linnik's theorem shows that the primes where it is slow are rare at every scale.

*References.* A. Tonelli, "Bemerkung über die Auflösung quadratischer Congruenzen", *Nachr. Ges. Wiss. Göttingen* (1891), 344–346. D. Shanks, "Five number-theoretic algorithms", *Proc. Second Manitoba Conf. Numer. Math.* (1973), 51–70. E. Bach, "Explicit bounds for primality testing and related problems", *Math. Comp.* 55 (1990), 355–380. P. Erdős, "Remarks on number theory I", *Mat. Lapok* 12 (1961), 10–16.